# Variance Partitioning — gpt2-xl L36 worddur
Unique semantic variance above and beyond lexical, syntactic, and acoustic controls.
`unique_semantic = R²_full − R²_controls`  (BH-FDR corrected per region × condition)

In [ ]:
import os, pickle, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

plt.rcParams.update({'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.dpi': 130})

VP_DIR  = '/scratch/aniluchavez/ConvoDATAS/VPResults/gpt2-xl_ctx200_worddur/pc100'
GLM_DIR = '/scratch/aniluchavez/ConvoDATAS/SemanticGLM/gpt2-xl_ctx200_worddur/pc100'
FIG_DIR = '../figures'
os.makedirs(FIG_DIR, exist_ok=True)

def load_all(pkl_dir, suffix):
    rows = []
    for f in sorted(glob.glob(os.path.join(pkl_dir, f'*{suffix}'))):
        obj = pickle.load(open(f, 'rb'))
        rows.append(obj['df'] if isinstance(obj, dict) else obj)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

vp  = load_all(VP_DIR,  '_VP.pkl')
glm = load_all(GLM_DIR, '_sem.pkl')

print('VP  shape:', vp.shape,  '| patients:', vp['patient'].nunique())
print('GLM shape:', glm.shape, '| patients:', glm['patient'].nunique())
print('VP columns:', list(vp.columns))

In [ ]:
# ── Summary table ──────────────────────────────────────────────────────────
rows = []
for region in vp['region'].unique():
    for cond in sorted(vp['condition'].unique()):
        sub = vp[(vp['region']==region) & (vp['condition']==cond)]
        sig = sub[sub['significant']]
        rows.append({
            'region': region, 'condition': cond,
            'n_neurons': len(sub),
            'n_sig_unique_sem': len(sig),
            'pct_sig': 100*len(sig)/len(sub) if len(sub) else np.nan,
            'med_unique_sem_all':  sub['unique_semantic'].median(),
            'med_unique_sem_sig':  sig['unique_semantic'].median() if len(sig) else np.nan,
            'med_r2_full_sig':     sig['r2_full'].median() if len(sig) else np.nan,
        })
display(pd.DataFrame(rows).round(4))

In [ ]:
# ── PLOT 1: Variance partitioning stacked bars by region × condition ───────
COMBOS = [('hippocampus','self'), ('hippocampus','other'), ('ACC','self'), ('ACC','other')]
PAL = {'unique_semantic': '#2166ac', 'unique_controls': '#d6604d', 'shared': '#92c5de'}
VP_COLS = ['unique_semantic', 'unique_controls', 'shared']

fig, axes = plt.subplots(1, 4, figsize=(13, 5), sharey=True)
for ax, (region, cond) in zip(axes, COMBOS):
    sub = vp[(vp['region']==region) & (vp['condition']==cond)]
    if sub.empty:
        ax.set_title(f'{region}/{cond}\n(no data)'); continue

    means = {c: sub[c].clip(lower=0).mean() for c in VP_COLS if c in sub.columns}
    pct_sig = 100 * sub['significant'].mean()

    bottom = 0
    for col in VP_COLS:
        v = means.get(col, 0)
        ax.bar(0, v, bottom=bottom, color=PAL[col], width=0.6)
        if v > 0.0005:
            ax.text(0, bottom + v/2, f'{v:.4f}',
                    ha='center', va='center', fontsize=8.5, color='white', fontweight='bold')
        bottom += v

    ax.set_title(f'{region}\n{cond}\n{pct_sig:.0f}% sig unique sem', fontsize=10)
    ax.set_xticks([])
    ax.set_xlim(-0.5, 0.5)

axes[0].set_ylabel('Mean R² (clipped ≥ 0)')
handles = [mpatches.Patch(facecolor=PAL[c], label=c.replace('_',' ')) for c in VP_COLS]
fig.legend(handles=handles, loc='lower center', ncol=3,
           fontsize=9, frameon=False, bbox_to_anchor=(0.5, -0.06))
plt.suptitle('gpt2-xl L36  |  5-fold temporal block CV  |  Variance partitioning', y=1.02)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/03_vp_bars.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── PLOT 2: % sig unique semantic per patient ───────────────────────────────
REGIONS = list(vp['region'].unique())
CONDS   = ['self', 'other']
COLORS  = {'self': '#2166ac', 'other': '#d6604d'}

fig, axes = plt.subplots(1, len(REGIONS), figsize=(6*len(REGIONS), 4.5), sharey=True)
if len(REGIONS) == 1: axes = [axes]

for ax, region in zip(axes, REGIONS):
    pats = sorted(vp['patient'].unique())
    xs = np.arange(len(pats))
    for i, cond in enumerate(CONDS):
        pcts = []
        for pat in pats:
            sub = vp[(vp['patient']==pat) & (vp['region']==region) & (vp['condition']==cond)]
            pcts.append(100*sub['significant'].mean() if len(sub) else np.nan)
        offset = (i - 0.5) * 0.25
        ax.scatter(xs+offset, pcts, color=COLORS[cond], s=60, label=cond, zorder=3)
        ax.plot(xs+offset, pcts, color=COLORS[cond], alpha=0.3, lw=1)
    ax.axhline(5, color='gray', lw=0.8, linestyle='--', label='5% chance')
    ax.set_xticks(xs)
    ax.set_xticklabels([p.replace('_task','\n') for p in pats], fontsize=7, rotation=45, ha='right')
    ax.set_ylabel('% sig unique semantic'); ax.set_title(region)
    ax.set_ylim(0, 105); ax.legend(fontsize=9)

plt.suptitle('% neurons with significant unique semantic variance  |  FDR q<0.05', y=1.02)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/03_vp_pct_sig_per_patient.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── PLOT 3: unique_semantic distribution — sig vs non-sig neurons ───────────
fig, axes = plt.subplots(1, len(REGIONS), figsize=(5*len(REGIONS), 4.5), sharey=False)
if len(REGIONS) == 1: axes = [axes]

for ax, region in zip(axes, REGIONS):
    for cond in CONDS:
        sub = vp[(vp['region']==region) & (vp['condition']==cond)]
        sig = sub[sub['significant']]['unique_semantic']
        ns  = sub[~sub['significant']]['unique_semantic']
        c = COLORS[cond]
        ax.hist(sig.clip(-0.05, 0.3), bins=40, alpha=0.6, color=c,
                label=f'{cond} sig (n={len(sig)})', density=True)
        ax.axvline(sig.median(), color=c, lw=1.5, linestyle='--')
    ax.axvline(0, color='k', lw=0.8, linestyle=':')
    ax.set_xlabel('Unique semantic R²')
    ax.set_ylabel('Density')
    ax.set_title(region)
    ax.legend(fontsize=9)

plt.suptitle('Unique semantic R² distribution (significant neurons only)', y=1.02)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/03_vp_unique_sem_dist.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── PLOT 4: GLM sig vs VP unique-sem sig overlap ────────────────────────────
# For each region × condition: what fraction of GLM-significant neurons also
# show significant unique semantic variance (above controls)?
fig, axes = plt.subplots(1, len(REGIONS), figsize=(6*len(REGIONS), 4.5))
if len(REGIONS) == 1: axes = [axes]

for ax, region in zip(axes, REGIONS):
    x_pos, x_labels, bar_colors = [], [], []
    pos = 0
    for cond in CONDS:
        g = glm[(glm['region']==region) & (glm['condition']==cond)]
        v = vp[(vp['region']==region)  & (vp['condition']==cond)]
        merged = pd.merge(
            g[['patient','neuron_idx','significant']].rename(columns={'significant':'sig_glm'}),
            v[['patient','neuron_idx','significant']].rename(columns={'significant':'sig_vp'}),
            on=['patient','neuron_idx'], how='inner'
        )
        if merged.empty: continue
        n = len(merged)
        both     = (merged['sig_glm'] & merged['sig_vp']).sum()
        glm_only = (merged['sig_glm'] & ~merged['sig_vp']).sum()
        vp_only  = (~merged['sig_glm'] & merged['sig_vp']).sum()
        neither  = (~merged['sig_glm'] & ~merged['sig_vp']).sum()

        vals   = [100*both/n, 100*glm_only/n, 100*vp_only/n, 100*neither/n]
        labels = ['both sig', 'GLM only', 'VP only', 'neither']
        clrs   = [COLORS[cond], COLORS[cond], '#888888', '#dddddd']
        alphas = [0.9, 0.5, 0.5, 0.3]

        for i, (val, lbl, clr, alph) in enumerate(zip(vals, labels, clrs, alphas)):
            ax.bar(pos, val, color=clr, alpha=alph, width=0.7,
                   label=f'{cond} {lbl}' if region == REGIONS[0] else None)
            if val > 2:
                ax.text(pos, val+0.5, f'{val:.0f}%', ha='center', fontsize=7.5)
            pos += 1
        ax.axvline(pos - 0.5, color='gray', lw=0.5, linestyle=':')
        pos += 0.5

    ax.set_ylabel('% of all neurons')
    ax.set_title(region)
    ax.set_xticks([])

# Manual legend
from matplotlib.patches import Patch
leg_handles = [
    Patch(facecolor=COLORS['self'],  alpha=0.9, label='self — both sig'),
    Patch(facecolor=COLORS['self'],  alpha=0.5, label='self — GLM only'),
    Patch(facecolor=COLORS['other'], alpha=0.9, label='other — both sig'),
    Patch(facecolor=COLORS['other'], alpha=0.5, label='other — GLM only'),
    Patch(facecolor='#888888',       alpha=0.5, label='VP only'),
    Patch(facecolor='#dddddd',       alpha=0.3, label='neither'),
]
fig.legend(handles=leg_handles, loc='lower center', ncol=3,
           fontsize=8, frameon=False, bbox_to_anchor=(0.5, -0.12))
plt.suptitle('Overlap: GLM sig (semantics) vs VP sig (unique semantic above controls)', y=1.02)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/03_glm_vp_overlap.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── PLOT 5: Self vs Other unique_semantic (sig neurons, violin) ─────────────
for region in REGIONS:
    self_u  = vp[(vp['region']==region) & (vp['condition']=='self')  & vp['significant']]['unique_semantic']
    other_u = vp[(vp['region']==region) & (vp['condition']=='other') & vp['significant']]['unique_semantic']

    fig, ax = plt.subplots(figsize=(5, 4.5))
    parts = ax.violinplot([self_u.dropna(), other_u.dropna()],
                           positions=[0, 1], showmedians=True)
    for pc, c in zip(parts['bodies'], [COLORS['self'], COLORS['other']]):
        pc.set_facecolor(c); pc.set_alpha(0.65)
    parts['cmedians'].set_color('k'); parts['cmedians'].set_linewidth(2)
    for key in ['cbars','cmins','cmaxes']:
        parts[key].set_color('k')

    _, p = stats.mannwhitneyu(self_u.dropna(), other_u.dropna(), alternative='two-sided')
    ax.set_xticks([0,1]); ax.set_xticklabels(['self','other'])
    ax.set_ylabel('Unique semantic R²')
    ax.set_title(f'{region}  |  sig neurons  |  Mann-Whitney p={p:.3g}')
    ax.axhline(0, color='gray', lw=0.8, linestyle='--')
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/03_vp_selfother_violin_{region}.pdf', bbox_inches='tight')
    plt.show()
    print(f'{region}: self med={self_u.median():.4f} (n={len(self_u)})  '
          f'other med={other_u.median():.4f} (n={len(other_u)})  p={p:.3g}')